
# EDA de Qualidade dos Dados — ENEM (sem filtros do `prepare_enem.py`)

Este notebook foi gerado como **template** para avaliar a **qualidade dos dados** dos microdados do ENEM **antes** de qualquer tratamento/limpeza do seu `prepare_enem.py`.

> **Como usar**
> 1. Ajuste o caminho do arquivo bruto (`RAW_PATH`) na célula de parâmetros.
> 2. Execute as células na ordem.  
> 3. Use os relatórios/saídas para escrever a seção **Qualidade dos Dados** do relatório (ausentes, inconsistências, duplicatas, ranges inválidos etc.).
>
> **Regras deste template**
> - Não aplica filtros/limpezas do seu pipeline.
> - Usa **pandas** e **matplotlib** (sem seaborn).
> - Pode trabalhar com **amostra** para agilidade em datasets muito grandes.


In [15]:
RAW_PATH = "../data/interim/unzipped_2023/DADOS/MICRODADOS_ENEM_2023.csv"
ENCODING = "latin1"     # ou "ISO-8859-1"
SEP = ";"               # arquivo usa ponto-e-vírgula
LOW_MEMORY = True
USE_SAMPLE = True                             # defina False para carregar todo o arquivo (risco de memória)
SAMPLE_FRAC = 0.02                            # fração da amostra (ex.: 2%)
RANDOM_STATE = 42
DECIMAL = ','                                  # decimal usado no CSV (ex.: '0,01')
ON_BAD_LINES = 'warn'                          # 'warn' para logar linhas ruins, 'skip' para pular
ENGINE = 'python'                              # parser mais tolerante quando necessário

In [16]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from textwrap import shorten
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


In [ ]:

# === Carregamento do dado bruto ===
path = Path(RAW_PATH)
assert path.exists(), f"Arquivo não encontrado: {path}"

# Prepara argumentos reutilizáveis para leitura
read_kwargs = dict(sep=SEP, encoding=ENCODING, decimal=DECIMAL, engine=ENGINE, low_memory=LOW_MEMORY, on_bad_lines=ON_BAD_LINES)

# 1) Tenta leitura direta (rápida). Se falhar, aplica fallbacks mais permissivos.
try:
    df_full = pd.read_csv(path, **read_kwargs)
    print("Leitura completa sem erros:", df_full.shape)
except Exception as e:
    print("Erro ao ler arquivo diretamente:", type(e).__name__, e)
    print("Tentando leitura permissiva: pulando linhas ruins e sem quoting.")
    import csv as _csv
    read_kwargs2 = read_kwargs.copy()
    # fallback: pula linhas malformadas e trata aspas/escaping de forma permissiva
    read_kwargs2.update(dict(on_bad_lines='skip', quoting=_csv.QUOTE_NONE, escapechar='\\'))
    try:
        df_full = pd.read_csv(path, **read_kwargs2)
        print("Leitura com fallback (linhas puladas):", df_full.shape)
    except Exception as e2:
        print("Fallback simples falhou:", type(e2).__name__, e2)
        print("Tentando leitura em chunks com o mesmo fallback (útil para arquivos grandes).")
        # tenta ler por chunks (salva os primeiros chunks para inspeção)
        chunks = pd.read_csv(path, **read_kwargs2, chunksize=100_000)
        dfs = []
        for i, ch in enumerate(chunks):
            print("chunk", i, "shape", ch.shape)
            dfs.append(ch)
            # limite para não consumir toda a memória aqui; aumente se quiser
            if i >= 9:
                break
        if dfs:
            df_full = pd.concat(dfs, ignore_index=True)
            print("Concat dos primeiros chunks:", df_full.shape)
        else:
            raise RuntimeError("Não foi possível ler sequer um chunk do arquivo")

# 2) Aplicar amostragem (se configurado) ou usar dataframe completo
if USE_SAMPLE and len(df_full) > 100_000:
    df = df_full.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE).copy()
    print(f"Amostra selecionada: {len(df)} de {len(df_full)} linhas ({SAMPLE_FRAC*100:.1f}%).")
else:
    df = df_full.copy()
    print(f"Dataset carregado (usando todos os dados lidos): {len(df)} linhas.")

print("Dimensões:", df.shape)
df.head(3)


Amostra selecionada: 78679 de 3933955 linhas (2.0%).
Dimensões: (78679, 76)


,NU_INSCRICAO,NU_ANO,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,NO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,CO_PROVA_CN,CO_PROVA_CH,CO_PROVA_LC,CO_PROVA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,TX_RESPOSTAS_CN,TX_RESPOSTAS_CH,TX_RESPOSTAS_LC,TX_RESPOSTAS_MT,TP_LINGUA,TX_GABARITO_CN,TX_GABARITO_CH,TX_GABARITO_LC,TX_GABARITO_MT,TP_STATUS_REDACAO,NU_NOTA_COMP1,NU_NOTA_COMP2,NU_NOTA_COMP3,NU_NOTA_COMP4,NU_NOTA_COMP5,NU_NOTA_REDACAO,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
553635,210059583535,2023,3,F,1,1,1,2,0,2,2.0,0,3303302.0,Niterói,33.0,RJ,2.0,1.0,1.0,3303302,Niterói,33,RJ,1,1,1,1,1221.0,1193.0,1204.0,1211.0,408.9,526.9,417.9,416.5,DDABBCCACBCCEDDBBBEBEBCDDCEACDCABBADDCACCDDCB,ACCBCABADAAAADEAECDABCACDBCACDBBADCCABCDEEBCD,BBCDACBCCDCADAAAEBBBDBDBEBBCCBBCCEDCBADBDDAED,CECCDEACDDCBDBCBCBBADDDDCEADCBBABBCDDAEBABCBA,0,DBEABDABDCACDBECDDDBCAAABBACCCADEBECCCEDAEEED,ACEEABAADCDAADEABCDABCDCABCBDADEBAECABADBCDAE,DBABBAEBAAAACDACDEDAACADBADBCCEACCCEAAECBBEBCA...,BCCDEEABCBEDCEABBEBDABDDADDADECAADDCCBEBEABCC,1.0,120.0,200.0,100.0,140.0,160.0,720.0,C,C,B,B,3,B,A,B,B,A,A,B,A,A,A,A,A,A,B,A,A,B,A,A,B
3349365,210060187802,2023,2,F,1,1,1,3,0,1,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3502804,Araçatuba,35,SP,1,1,1,1,1224.0,1192.0,1202.0,1214.0,499.4,535.6,549.2,570.7,ECADAEACBDBDABDDEBEBACBBDDECBBCDDBECBDEDCBCBA,DDAADEACCDBCBDCCEBEBCBDDAEDDCBECBCDBAEDDBEBCD,ABBCCEBEAEDDABBBACDBBAACACAEABBEEBBDBDCDADCAC,EABADDBECBCCAEBDEAAEDEAACDABCDCDBDEECDEABDDCA,1,CDDDABBABDBEABDECCEEEDCEDAEBABDCCAACCCADACDBE,DBAADEADCDCABABCDDEBAEABAECABAACECDAECBDAABCD,BBBDAABAEACCEEEDEACBCACAACAACAAAECBBEDBCCADBDE...,EBDADDAEBEACBEDCECCBEABCADEBCCBCCDEBDDAABBADD,1.0,120.0,120.0,120.0,120.0,80.0,560.0,F,F,D,B,3,D,D,C,C,B,C,B,B,B,A,B,A,B,C,A,A,D,B,C,B
2297393,210059061595,2023,8,M,1,2,1,1,0,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2103703,Cururupu,21,MA,1,1,1,1,1224.0,1192.0,1202.0,1214.0,425.2,391.7,446.0,503.5,EDDACBCBBDBAEBDEBBACBEBACBBECCBAAACCCEEACCEDA,DEBDACCBDDBAEBBAECBDDBCDCCDDEABACAEBEBCCEABCE,DBAECCEEADBADEBBDBCECACDBAADCEBDBCCADDBDBEECB,CDEABEBEBABDDDDCBCDABCEBDECBADDCADCBCDBBCDADB,0,CDDDABBABDBEABDECCEEEDCEDAEBABDCCAACCCADACDBE,DBAADEADCDCABABCDDEBAEABAECABAACECDAECBDAABCD,BBBDAABAEACCEEEDEACBCACAACAACAAAECBBEDBCCADBDE...,EBDADDAEBEACBEDCECCBEABCADEBCCBCCDEBDDAABBADD,1.0,80.0,100.0,100.0,100.0,80.0,460.0,D,C,A,A,2,B,A,B,D,A,A,B,A,A,A,A,A,A,B,A,B,D,A,A,B


In [14]:

# === Esquema rápido e uso de memória ===
info_buf = []
df.info(buf=info_buf := [])
# pandas não retorna string diretamente no info(); vamos usar o display padrão
df.info()
print("\nUso de memória (aprox.):")
print(df.memory_usage(deep=True).sum() / (1024**2), "MB")


SyntaxError: invalid syntax (1782257973.py, line 3)

In [ ]:

# === Completude (ausentes) ===
missing_counts = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing_counts / len(df)).sort_values(ascending=False)
missing = pd.DataFrame({"missing_count": missing_counts, "missing_pct": missing_pct})
display(missing.head(25))

print("\nColunas com > 50% ausentes:")
display(missing[missing["missing_pct"] > 0.50])


In [ ]:

# === Duplicatas ===
dup_rows = df.duplicated().sum()
print(f"Linhas completamente duplicadas: {dup_rows}")
# Se houver um identificador único (ex.: 'NU_INSCRICAO'), substitua abaixo
CANDIDATE_KEYS = []  # ex.: ["NU_INSCRICAO"]
for key in CANDIDATE_KEYS:
    if key in df.columns:
        dups = df.duplicated(subset=[key]).sum()
        print(f"Duplicatas por '{key}': {dups}")


In [ ]:

# === Sanidade numérica (ex.: notas ENEM) ===
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
summary = df[numeric_cols].describe().T
summary["missing_pct"] = df[numeric_cols].isna().mean()
display(summary.head(20))

# Regras simples de validação para algumas colunas comuns do ENEM (ajuste conforme disponível no seu CSV)
rules = {
    "NU_NOTA_MT": (0, 1000),
    "NU_NOTA_LC": (0, 1000),
    "NU_NOTA_CH": (0, 1000),
    "NU_NOTA_CN": (0, 1000),
    "NU_NOTA_REDACAO": (0, 1000),
}
violations = {}
for col, (lo, hi) in rules.items():
    if col in df.columns:
        mask = (~df[col].isna()) & ((df[col] < lo) | (df[col] > hi))
        violations[col] = int(mask.sum())
violations


In [ ]:

# === Auditoria de categóricas ===
# Mostra top categorias por coluna e possíveis placeholders ('99','-1','NA','Não informado')
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

def sample_value_counts(s, top=10):
    vc = s.value_counts(dropna=False).head(top)
    vc = vc.reset_index().rename(columns={"index": "valor", s.name: "contagem"})
    vc["valor"] = vc["valor"].astype(str).apply(lambda x: shorten(x, width=60, placeholder="…"))
    return vc

placeholders = {"99", "98", "97", "-1", "NA", "N/A", "NaN", "Não informado", "Ignorado"}

report = {}
for col in cat_cols[:20]:  # limita para visual
    vc = df[col].astype(str).str.strip()
    ph_hits = vc.isin(placeholders).mean()
    top = sample_value_counts(vc, top=8)
    report[col] = {"placeholder_rate": ph_hits, "top_values": top}

# Exibe um pequeno sumário das primeiras colunas categóricas
from IPython.display import display
for col, stats in list(report.items())[:10]:
    print(f"\n--- {col} ---")
    print(f"Placeholders (fração): {stats['placeholder_rate']:.3f}")
    display(stats["top_values"])


In [ ]:

# === Padrões de ausência por grupo (ex.: por tipo de escola) ===
GROUP_COL = None  # ex.: "TP_ESCOLA"
TARGET_COLS = ["NU_NOTA_MT", "NU_NOTA_LC", "NU_NOTA_CH", "NU_NOTA_CN", "NU_NOTA_REDACAO"]

if GROUP_COL and GROUP_COL in df.columns:
    for col in TARGET_COLS:
        if col in df.columns:
            pct = df.groupby(GROUP_COL)[col].apply(lambda s: s.isna().mean()).sort_values(ascending=False)
            print(f"Ausência relativa por {GROUP_COL} → {col}")
            display(pct)
else:
    print("Defina GROUP_COL (ex.: 'TP_ESCOLA') para ver padrão de ausentes por grupo.")


In [ ]:

# === Visuais leves (sem seaborn) ===
cols_to_plot = [c for c in ["NU_NOTA_MT", "NU_NOTA_LC", "NU_NOTA_CH", "NU_NOTA_CN", "NU_NOTA_REDACAO"] if c in df.columns]

for col in cols_to_plot:
    series = df[col].dropna()
    if len(series) == 0:
        continue
    plt.figure()
    plt.hist(series, bins=40)
    plt.title(f"Histograma — {col}")
    plt.xlabel(col); plt.ylabel("Frequência")
    plt.show()

# Boxplot simples (se útil)
if cols_to_plot:
    plt.figure()
    data = [df[c].dropna().values for c in cols_to_plot]
    plt.boxplot(data, labels=cols_to_plot, showfliers=True)
    plt.title("Boxplots — Notas")
    plt.ylabel("Valor")
    plt.show()


In [ ]:

# === Exports úteis ===
out_dir = Path("/mnt/data/quality_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

missing.to_csv(out_dir / "missing_by_column.csv")
summary.to_csv(out_dir / "numeric_summary.csv")

print("Arquivos salvos em:", out_dir)
